In [2]:
"""
회의실 예약 자동화 스크립트 v7

동작 방식:
  1. primary 캘린더에서 이번 주 일정 전체 조회
  2. 각 이벤트의 attendees 이메일 → TEAM_MAP으로 팀 분류 (하드코딩)
     - TEAM_MAP에 없는 이메일은 무시 (본인 포함)
     - 여러 팀 이메일이 섞인 이벤트 → 해당 팀 모두에 표시
     - 팀원이 한 명도 없는 이벤트 → 문자에 미포함
  3. 이미 예약된 회의실은 resource attendee displayName에서 파싱
     - 복수 회의실이면 [마포 18-A / 선릉 4-C] 형태로 합산
  4. 팀별로 묶어 안내 문자 형식으로 출력
  5. 회의실 미배정 이벤트에 리소스 자동 예약

사전 준비:
    pip install google-auth google-auth-oauthlib google-auth-httplib2 google-api-python-client

Google Cloud Console 설정:
    1. https://console.cloud.google.com → 프로젝트 생성
    2. Google Calendar API 활성화
    3. OAuth 2.0 클라이언트 ID 생성 (데스크톱 앱)
    4. credentials.json 다운로드 → 스크립트와 같은 폴더에 저장
"""

import os
import json
import re
from datetime import datetime, timedelta
from zoneinfo import ZoneInfo
from collections import defaultdict
from google.oauth2.credentials import Credentials
from google_auth_oauthlib.flow import InstalledAppFlow
from google.auth.transport.requests import Request
from googleapiclient.discovery import build

# ─────────────────────────────────────────
# 설정
# ─────────────────────────────────────────

SCOPES = ["https://www.googleapis.com/auth/calendar"]
CREDENTIALS_FILE = "credentials.json"
TOKEN_FILE = "token.json"
TZ = ZoneInfo("Asia/Seoul")
WEEKDAY_KO = ["월", "화", "수", "목", "금", "토", "일"]
TEAM_ORDER = ["투자전략팀", "펀드팀", "투자팀"]

# ─────────────────────────────────────────
# ★ 팀원 이메일 → 팀 하드코딩 매핑
#   본인(seokjoo) 제외
#   이벤트 attendees에 해당 이메일이 있어야만 팀 분류됨
# ─────────────────────────────────────────

TEAM_MAP: dict[str, str] = {
    # 투자전략팀
    "hanui@dcamp.kr":       "투자전략팀",
    "hyohyun@dcamp.kr":     "투자전략팀",
    "juheum@dcamp.kr":      "투자전략팀",
    # 펀드팀
    "myeongcheol@dcamp.kr": "펀드팀",
    "jiyoungheo@dcamp.kr":  "펀드팀",
    "sanghyeok@dcamp.kr":   "펀드팀",
    "ykpark@dcamp.kr":      "펀드팀",
    "seoha@dcamp.kr":       "펀드팀",
    "jiny@dcamp.kr":        "펀드팀",
    # 투자팀
    "youngsang@dcamp.kr":   "투자팀",
    "jinha@dcamp.kr":       "투자팀",
    "hansol.kim@dcamp.kr":  "투자팀",
    "nakhwan@dcamp.kr":     "투자팀",
    "eunjeong@dcamp.kr":    "투자팀",
    "inho@dcamp.kr":        "투자팀",
    "gayoung@dcamp.kr":     "투자팀",
}

# ─────────────────────────────────────────
# 회의실 리소스 ID 매핑
# 워크스페이스 관리자 콘솔 → 디렉터리 → 건물 및 리소스 에서 확인
# ─────────────────────────────────────────

ROOM_RESOURCES: dict[str, str] = {
    # "선릉 2-B": "c_188fjm7paiu72ihgmbq91mj5mgak6@resource.calendar.google.com",
    # "선릉 2-C": "c_xxx@resource.calendar.google.com",
    # "선릉 4-B": "c_18825va2grf3oi1rlbnjbilk95ju6@resource.calendar.google.com",
    # "선릉 4-C": "c_1885df2a2fojiiqamt7q85ctj2ib4@resource.calendar.google.com",
    # "마포 18-A": "c_1882is5r6jjlkicektsrete5on4uk@resource.calendar.google.com",
    # "마포 18-B": "c_1888rnc3fr0qkg26hr13vj39th48e@resource.calendar.google.com",
}


# ─────────────────────────────────────────
# 인증
# ─────────────────────────────────────────

def authenticate():
    creds = None
    if os.path.exists(TOKEN_FILE):
        creds = Credentials.from_authorized_user_file(TOKEN_FILE, SCOPES)
    if not creds or not creds.valid:
        if creds and creds.expired and creds.refresh_token:
            creds.refresh(Request())
        else:
            flow = InstalledAppFlow.from_client_secrets_file(CREDENTIALS_FILE, SCOPES)
            creds = flow.run_local_server(port=0)
        with open(TOKEN_FILE, "w") as f:
            f.write(creds.to_json())
    return build("calendar", "v3", credentials=creds)


# ─────────────────────────────────────────
# 1단계: 일정 조회
# ─────────────────────────────────────────

def get_week_range():
    today = datetime.now(TZ)
    monday = (today - timedelta(days=today.weekday())).replace(
        hour=0, minute=0, second=0, microsecond=0
    )
    sunday = monday + timedelta(days=6, hours=23, minutes=59, seconds=59)
    return monday, sunday


def fetch_all_team_events(service) -> list:
    time_min, time_max = get_week_range()
    print(f"\n📅 조회 기간: {time_min.strftime('%Y-%m-%d')} ~ {time_max.strftime('%Y-%m-%d')}")

    # 이벤트 ID 기준으로 중복 제거하면서 수집
    # 각 이벤트에 어느 팀원 캘린더에서 가져왔는지 태깅
    event_map: dict[str, dict] = {}  # event_id → event

    for email in TEAM_MAP.keys():
        try:
            page_token = None
            while True:
                result = service.events().list(
                    calendarId=email,
                    timeMin=time_min.isoformat(),
                    timeMax=time_max.isoformat(),
                    singleEvents=True,
                    orderBy="startTime",
                    maxResults=250,
                    pageToken=page_token,
                ).execute()
                for ev in result.get("items", []):
                    eid = ev.get("id", "")
                    if eid not in event_map:
                        event_map[eid] = ev
                        event_map[eid]["_team_emails"] = set()
                    event_map[eid]["_team_emails"].add(email)
                page_token = result.get("nextPageToken")
                if not page_token:
                    break
            print(f"  ✅ {email}")
        except Exception as e:
            print(f"  ❌ {email}: {e}")

    events = list(event_map.values())
    print(f"  → 총 {len(events)}개 고유 이벤트 수집됨")
    return events


# ─────────────────────────────────────────
# 2단계: 파싱 & 팀 분류
# ─────────────────────────────────────────

def parse_event_time(event: dict):
    s = event["start"].get("dateTime")
    e = event["end"].get("dateTime")
    if not s or not e:
        return None, None
    return (
        datetime.fromisoformat(s).astimezone(TZ),
        datetime.fromisoformat(e).astimezone(TZ),
    )


def clean_room_name(raw: str) -> str:
    """'Taap-선릉-4-4-B (6)' → '선릉 4-B'"""
    m = re.search(r"Taap[-_]?(선릉|마포)[-_](\d+)[-_]\d+[-_]([A-Za-z])", raw, re.IGNORECASE)
    if m:
        return f"{m.group(1)} {m.group(2)}-{m.group(3).upper()}"
    m2 = re.search(r"(선릉|마포)\s*(\d+[-]\s*[A-Za-z])", raw)
    if m2:
        return f"{m2.group(1)} {m2.group(2)}"
    return raw.strip()


def get_booked_rooms(event: dict) -> list[str]:
    """이벤트에 예약된 회의실 이름 목록 반환 (복수 가능)"""
    rooms = []
    for att in event.get("attendees", []):
        email = att.get("email", "")
        if "@resource.calendar.google.com" not in email:
            continue
        matched = next(
            (name for name, rid in ROOM_RESOURCES.items() if rid == email), None
        )
        if matched:
            rooms.append(matched)
        else:
            display = att.get("displayName", "")
            if display:
                rooms.append(clean_room_name(display))
    # location fallback
    if not rooms:
        location = event.get("location", "")
        if location:
            rooms.append(clean_room_name(location))
    return rooms


def get_teams_for_event(event: dict) -> set[str]:
    teams: set[str] = set()
    # 직접 조회한 캘린더 이메일 기준
    for email in event.get("_team_emails", set()):
        if email in TEAM_MAP:
            teams.add(TEAM_MAP[email])
    return teams

def classify_events_by_team(events: list) -> dict:
    by_date_team: dict = defaultdict(lambda: defaultdict(list))

    for ev in events:
        start, end = parse_event_time(ev)
        if not start:
            continue

        # 토요일(5), 일요일(6) 제외
        if start.weekday() >= 5:
            continue

        # 점심시간(12:00~13:00) 포함 일정 제외
        lunch_start = start.replace(hour=12, minute=0, second=0, microsecond=0)
        lunch_end = start.replace(hour=13, minute=0, second=0, microsecond=0)
        if start < lunch_end and end > lunch_start:
            continue

        teams = get_teams_for_event(ev)
        if not teams:
            continue

        date_key = start.strftime("%Y-%m-%d")
        entry = {
            "id":    ev.get("id", ""),
            "title": ev.get("summary", "(제목 없음)").strip(),
            "start": start,
            "end":   end,
            "rooms": get_booked_rooms(ev),
        }
        for team in teams:
            by_date_team[date_key][team].append(entry)

    return by_date_team


# ─────────────────────────────────────────
# 3단계: 안내 문자 생성
# ─────────────────────────────────────────

def build_sms(date_str: str, team_events: dict) -> str:
    date = datetime.strptime(date_str, "%Y-%m-%d").replace(tzinfo=TZ)
    weekday = WEEKDAY_KO[date.weekday()]
    lines = [f"💡 {date.month}/{date.day} {weekday}요일, 회의실 안내드립니다!"]

    for team in TEAM_ORDER:
        evs = team_events.get(team)
        if not evs:
            continue

        seen: set = set()
        unique_evs = [e for e in evs if not (e["id"] in seen or seen.add(e["id"]))]
        unique_evs.sort(key=lambda x: x["start"])

        lines.append(f"□ {team}")
        for ev in unique_evs:
            room_str = " / ".join(ev["rooms"]) if ev["rooms"] else "미배정"
            t = f"{ev['start'].strftime('%H:%M')}~{ev['end'].strftime('%H:%M')}"
            lines.append(f"[{room_str}] {t} {ev['title']}")

    lines.append("추가 일정이나 변경 사항 있으시면 반영하겠습니다. 감사합니다.")
    return "\n".join(lines)


def generate_weekly_sms(by_date_team: dict) -> dict[str, str]:
    return {
        date_str: build_sms(date_str, team_events)
        for date_str, team_events in sorted(by_date_team.items())
    }


# ─────────────────────────────────────────
# 4단계: 회의실 자동 예약
# ─────────────────────────────────────────

def list_rooms(service) -> dict:
    rooms, page_token = {}, None
    while True:
        result = service.calendarList().list(pageToken=page_token).execute()
        for cal in result.get("items", []):
            cal_id = cal.get("id", "")
            if "@resource.calendar.google.com" in cal_id:
                raw = cal.get("summaryOverride") or cal.get("summary", cal_id)
                rooms[clean_room_name(raw)] = cal_id
        page_token = result.get("nextPageToken")
        if not page_token:
            break
    return rooms


def is_room_available(service, room_cal_id: str, start: datetime, end: datetime) -> bool:
    try:
        result = service.events().list(
            calendarId=room_cal_id,
            timeMin=start.isoformat(),
            timeMax=end.isoformat(),
            singleEvents=True,
        ).execute()
        for ev in result.get("items", []):
            s = ev["start"].get("dateTime")
            e = ev["end"].get("dateTime")
            if not s:
                continue
            if datetime.fromisoformat(s).astimezone(TZ) < end and \
               datetime.fromisoformat(e).astimezone(TZ) > start:
                return False
        return True
    except Exception:
        return False


def book_room(service, entry: dict, room_name: str, room_cal_id: str) -> bool:
    if not is_room_available(service, room_cal_id, entry["start"], entry["end"]):
        print(f"    ❌ {room_name}: 이미 예약됨")
        return False
    try:
        cal_event = service.events().get(calendarId="primary", eventId=entry["id"]).execute()
        attendees = cal_event.get("attendees", [])
        if any(a.get("email") == room_cal_id for a in attendees):
            print(f"    ✅ {room_name}: 이미 추가됨")
            return True
        attendees.append({"email": room_cal_id, "displayName": room_name})
        service.events().patch(
            calendarId="primary",
            eventId=entry["id"],
            body={"attendees": attendees},
            sendUpdates="none",
        ).execute()
        print(f"    ✅ {room_name}: 예약 완료!")
        entry["rooms"].append(room_name)
        return True
    except Exception as e:
        print(f"    ❌ 예약 실패: {e}")
        return False


def auto_book_all(service, by_date_team: dict) -> dict:
    rooms = ROOM_RESOURCES.copy()
    if not rooms:
        print("\n⚠️  ROOM_RESOURCES 비어있어 자동 조회 시도...")
        rooms = list_rooms(service)
        if rooms:
            print(f"   발견된 회의실: {list(rooms.keys())}")
            ROOM_RESOURCES.update(rooms)
        else:
            print("   ⚠️  회의실 없음. 스크립트 상단 ROOM_RESOURCES를 직접 입력하세요.")
            return {}

    booked: dict = {}
    print(f"\n{'─'*60}\n🏢 회의실 자동 예약\n{'─'*60}")

    seen_ids: set = set()
    for date_str in sorted(by_date_team.keys()):
        for team, evs in by_date_team[date_str].items():
            for ev in evs:
                eid = ev["id"]
                if eid in seen_ids or ev["rooms"]:
                    continue
                seen_ids.add(eid)
                print(f"\n  [{ev['title']}] {ev['start'].strftime('%m/%d %H:%M')}~{ev['end'].strftime('%H:%M')}")
                for room_name, room_cal_id in rooms.items():
                    if book_room(service, ev, room_name, room_cal_id):
                        booked[eid] = room_name
                        break
    return booked


# ─────────────────────────────────────────
# 메인
# ─────────────────────────────────────────

def main():
    print("=" * 60)
    print("  📆 회의실 예약 자동화 v7  ")
    print("=" * 60)

    print("\n🔐 Google 인증 중...")
    service = authenticate()
    print("  ✅ 인증 완료")

    print("\n[1단계] 이번 주 일정 조회")
    events = fetch_all_team_events(service)

    print("\n[2단계] 팀별 일정 분류")
    by_date_team = classify_events_by_team(events)
    total = sum(len(evs) for dt in by_date_team.values() for evs in dt.values())
    print(f"  → {len(by_date_team)}일치 / {total}건 분류 완료")

    print("\n[3단계] 회의실 자동 예약")
    booked = auto_book_all(service, by_date_team)

    print(f"\n[4단계] 안내 문자 생성")
    messages = generate_weekly_sms(by_date_team)
    for date_str, msg in messages.items():
        print(f"\n{'━'*60}\n{msg}")

    output = {
        "generated_at": datetime.now(TZ).isoformat(),
        "booked_rooms":  booked,
        "sms_messages":  messages,
    }
    with open("meeting_report.json", "w", encoding="utf-8") as f:
        json.dump(output, f, ensure_ascii=False, indent=2, default=str)
    print(f"\n{'━'*60}")
    print("💾 결과 저장: meeting_report.json")
    print("✅ 완료!")


if __name__ == "__main__":
    main()

  📆 회의실 예약 자동화 v7  

🔐 Google 인증 중...
  ✅ 인증 완료

[1단계] 이번 주 일정 조회

📅 조회 기간: 2026-04-13 ~ 2026-04-19
  ✅ hanui@dcamp.kr
  ✅ hyohyun@dcamp.kr
  ✅ juheum@dcamp.kr
  ✅ myeongcheol@dcamp.kr
  ✅ jiyoungheo@dcamp.kr
  ✅ sanghyeok@dcamp.kr
  ✅ ykpark@dcamp.kr
  ✅ seoha@dcamp.kr
  ✅ jiny@dcamp.kr
  ✅ youngsang@dcamp.kr
  ✅ jinha@dcamp.kr
  ✅ hansol.kim@dcamp.kr
  ✅ nakhwan@dcamp.kr
  ✅ eunjeong@dcamp.kr
  ✅ inho@dcamp.kr
  ✅ gayoung@dcamp.kr
  → 총 227개 고유 이벤트 수집됨

[2단계] 팀별 일정 분류
  → 5일치 / 128건 분류 완료

[3단계] 회의실 자동 예약

⚠️  ROOM_RESOURCES 비어있어 자동 조회 시도...
   ⚠️  회의실 없음. 스크립트 상단 ROOM_RESOURCES를 직접 입력하세요.

[4단계] 안내 문자 생성

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
💡 4/13 월요일, 회의실 안내드립니다!
□ 투자전략팀
[미배정] 10:00~10:30 직접투자 파이프라인 브리핑
[미배정] 10:30~11:30 ✨D.MINGLE
[미배정] 14:00~14:30 (마포) 효현-지인 티타임
[마포 17-E] 14:30~15:30 (17E/온라인) 배치 8기 킥오프 미팅
[미배정] 16:00~17:00 마포) 레오스페이스 IR
[미배정] 17:00~18:00 [마포] ACVC 검토 보고
[미배정] 18:00~20:00 [공덕] 조물조물 가죽공방 (케이스/워치스트랩)
[미배정] 19:00~20:00 (제목 없음)
□ 펀드팀
[미배정] 10:

In [12]:
from datetime import datetime
from zoneinfo import ZoneInfo
from google.auth.transport.requests import Request
from google.oauth2.credentials import Credentials
from google_auth_oauthlib.flow import InstalledAppFlow
from google.auth.exceptions import RefreshError
from googleapiclient.discovery import build
import os

# Google Calendar API setup
SCOPES = ['https://www.googleapis.com/auth/calendar.readonly']

def authenticate_google_calendar():
    """Authenticate and return Google Calendar service"""
    creds = None
    
    if os.path.exists('token.json'):
        creds = Credentials.from_authorized_user_file('token.json', SCOPES)
    
    if not creds or not creds.valid:
        if creds and creds.expired and creds.refresh_token:
            try:
                creds.refresh(Request())
            except RefreshError:
                creds = None
        
        if not creds:
            flow = InstalledAppFlow.from_client_secrets_file(
                'credentials.json', SCOPES)
            creds = flow.run_local_server(port=0)
        
        with open('token.json', 'w') as token:
            token.write(creds.to_json())
    
    return build('calendar', 'v3', credentials=creds)

# Authenticate
service = authenticate_google_calendar()

TZ = ZoneInfo("Asia/Seoul")

# 각 캘린더별 확인 일정 설정
calendars = {
    "primary": {
        "name": "내 캘린더",
        "date_range": {
            "start": datetime(2026, 4, 17, 0, 0, tzinfo=TZ),
            "end": datetime(2026, 4, 18, 0, 0, tzinfo=TZ)
        }
    },
    "sanghyeok@dcamp.kr": {
        "name": "sanghyeok@dcamp.kr",
        "date_range": {
            "start": datetime(2026, 4, 24, 0, 0, tzinfo=TZ),
            "end": datetime(2026, 4, 25, 0, 0, tzinfo=TZ)
        }
    }
}

for cal_id, cal_info in calendars.items():
    print(f"\n{'='*50}")
    print(f"📅 {cal_info['name']}")
    print(f"확인 기간: {cal_info['date_range']['start'].date()} ~ {cal_info['date_range']['end'].date()}")
    print(f"{'='*50}\n")
    
    time_min = cal_info['date_range']['start'].isoformat()
    time_max = cal_info['date_range']['end'].isoformat()
    
    try:
        result = service.events().list(
            calendarId=cal_id,
            timeMin=time_min,
            timeMax=time_max,
            singleEvents=True,
            orderBy="startTime"
        ).execute()
        
        events = result.get("items", [])
        print(f"일정 {len(events)}개 찾음\n")
        
        if not events:
            print("일정이 없습니다.\n")
            continue
        
        for ev in events:
            event_id = ev.get("id")
            summary = ev.get("summary", "제목 없음")
            
            print(f"제목: {summary}")
            print(f"event_id: {event_id}")
            print(f"시작: {ev.get('start', {}).get('dateTime', ev.get('start', {}).get('date'))}")
            print(f"종료: {ev.get('end', {}).get('dateTime', ev.get('end', {}).get('date'))}")
            
            # 이벤트 상세 정보 조회 (회의실 리소스 포함)
            try:
                event_detail = service.events().get(
                    calendarId=cal_id,
                    eventId=event_id
                ).execute()
                
                # 회의실 리소스 추출
                attendees = event_detail.get("attendees", [])
                resources = [att for att in attendees if "@resource.calendar.google.com" in att.get("email", "")]
                
                if resources:
                    print("\n[회의실 리소스]")
                    for resource in resources:
                        print("  resource_email:", resource.get("email", ""))
                        print("  displayName   :", resource.get("displayName", ""))
                        print("  responseStatus:", resource.get("responseStatus", ""))
                        print("  " + "-" * 30)
                else:
                    print("\n[회의실 리소스] 없음")
                
            except Exception as e:
                print(f"❌ 이벤트 상세 조회 오류: {e}")
            
            print("-" * 50)
            
    except Exception as e:
        print(f"❌ 오류 발생: {e}\n")


📅 내 캘린더
확인 기간: 2026-04-17 ~ 2026-04-18

일정 5개 찾음

제목: 투자전략팀 고정 회의
event_id: 0c8k77s73qifbou99jn7ha6n52
시작: 2026-04-17T10:00:00+09:00
종료: 2026-04-17T11:00:00+09:00

[회의실 리소스]
  resource_email: c_1882is5r6jjlkicektsrete5on4uk@resource.calendar.google.com
  displayName   : Taap-마포-18-18-A (6)
  responseStatus: accepted
  ------------------------------
--------------------------------------------------
제목: 2-A
event_id: 1shaj700dkh20aa4c41jkcqs1o
시작: 2026-04-17T11:30:00+09:00
종료: 2026-04-17T12:00:00+09:00

[회의실 리소스]
  resource_email: c_188cephv2n692jtji6hv2svan5e60@resource.calendar.google.com
  displayName   : Taap-선릉-2-2-A (8)
  responseStatus: accepted
  ------------------------------
--------------------------------------------------
제목: 2-B
event_id: 2idm6huhgev1ect6n2qesvmnkq
시작: 2026-04-17T12:00:00+09:00
종료: 2026-04-17T12:30:00+09:00

[회의실 리소스]
  resource_email: c_188fjm7paiu72ihgmbq91mj5mgak6@resource.calendar.google.com
  displayName   : Taap-선릉-2-2-B (6)
  responseStatus: accept

In [5]:
import os
from datetime import datetime
from zoneinfo import ZoneInfo

from google.oauth2.credentials import Credentials
from google_auth_oauthlib.flow import InstalledAppFlow
from google.auth.transport.requests import Request
from googleapiclient.discovery import build

SCOPES = ["https://www.googleapis.com/auth/calendar"]
CREDENTIALS_FILE = "credentials.json"
TOKEN_FILE = "token.json"
TZ = ZoneInfo("Asia/Seoul")

ROOM_RESOURCE_EMAIL = "c_1882is5r6jjlkicektsrete5on4uk@resource.calendar.google.com"
ROOM_NAME = "마포 18-A"



테스트 이벤트 생성 완료
event_id: 0mcp1bsge7a0qj169bcnia0r2c

회의실 추가 후 attendees
c_1882is5r6jjlkicektsrete5on4uk@resource.calendar.google.com | displayName: 마포 18-A | responseStatus: needsAction

확인할 것
1. 내 캘린더에 [테스트] 마포 18-A 예약 확인 이벤트 생성 여부
2. 참석자에 마포 18-A 리소스가 들어갔는지
3. responseStatus가 accepted인지
4. 실제 마포 18-A 회의실 캘린더에도 예약이 잡혔는지
